# Employee Attrition Prediction — Feature Engineering & Preprocessing

This notebook prepares the cleaned employee attrition dataset for machine learning.

## Objectives

- Load the cleaned dataset
- Separate features and target
- Inspect numerical and categorical columns
- Create a train/test split using stratification
- Build numerical and categorical preprocessing pipelines
- Combine them with `ColumnTransformer`
- Fit preprocessing only on the training set
- Transform train and test data
- Inspect generated feature names
- Save the fitted preprocessor and split metadata

## Important Principle

The preprocessor is fitted only on the training data. This prevents information from the test set from leaking into model training.

In [1]:
from pathlib import Path
import json
import platform
import warnings

import joblib
import numpy as np
import pandas as pd
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 140)

RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET_COLUMN = "Attrition"

print("Libraries imported successfully.")
print("Python version:", platform.python_version())
print("scikit-learn version:", sklearn.__version__)

Libraries imported successfully.
Python version: 3.14.4
scikit-learn version: 1.9.0


## 1. Configure Project Paths

Expected project structure:

```text
employee-attrition-prediction/
├── data/
│   └── processed/
│       └── clean_employee_attrition.csv
├── models/
├── notebooks/
│   └── 03_feature_engineering_preprocessing.ipynb
└── reports/
```

In [2]:
CURRENT_DIR = Path.cwd()

PROJECT_ROOT = (
    CURRENT_DIR.parent
    if CURRENT_DIR.name == "notebooks"
    else CURRENT_DIR
)

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "clean_employee_attrition.csv"
)

MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"

PREPROCESSOR_PATH = MODELS_DIR / "preprocessor.joblib"
SPLIT_METADATA_PATH = REPORTS_DIR / "preprocessing_metadata.json"
FEATURE_NAMES_PATH = REPORTS_DIR / "processed_feature_names.csv"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT.resolve())
print("Dataset path:", DATA_PATH.resolve())
print("Preprocessor output:", PREPROCESSOR_PATH.resolve())

Project root: C:\Users\Nasteho Abdi\employee-attrition-prediction
Dataset path: C:\Users\Nasteho Abdi\employee-attrition-prediction\data\processed\clean_employee_attrition.csv
Preprocessor output: C:\Users\Nasteho Abdi\employee-attrition-prediction\models\preprocessor.joblib


## 2. Load the Cleaned Dataset

In [3]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Cleaned dataset not found at: {DATA_PATH.resolve()}\n"
        "Run Phase 3 first to create clean_employee_attrition.csv."
    )

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully.
Shape: (1470, 31)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,1,Travel_Rarely,1102,Sales,1,2,Life Sciences,2,Female,94,3,2,Sales Executive,4,Single,5993,19479,8,Yes,11,3,1,0,8,0,1,6,4,0,5
1,49,0,Travel_Frequently,279,Research & Development,8,1,Life Sciences,3,Male,61,2,2,Research Scientist,2,Married,5130,24907,1,No,23,4,4,1,10,3,3,10,7,1,7
2,37,1,Travel_Rarely,1373,Research & Development,2,2,Other,4,Male,92,2,1,Laboratory Technician,3,Single,2090,2396,6,Yes,15,3,2,0,7,3,3,0,0,0,0
3,33,0,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,4,Female,56,3,1,Research Scientist,3,Married,2909,23159,1,Yes,11,3,3,0,8,3,3,8,7,3,0
4,27,0,Travel_Rarely,591,Research & Development,2,1,Medical,1,Male,40,3,1,Laboratory Technician,2,Married,3468,16632,9,No,12,3,4,1,6,3,3,2,2,2,2


## 3. Validate the Dataset

In [4]:
if TARGET_COLUMN not in df.columns:
    raise ValueError(
        f"Target column '{TARGET_COLUMN}' was not found."
    )

if df.empty:
    raise ValueError("The dataset is empty.")

if df[TARGET_COLUMN].isnull().any():
    raise ValueError("The target column contains missing values.")

unexpected_target_values = set(df[TARGET_COLUMN].unique()) - {0, 1}

if unexpected_target_values:
    raise ValueError(
        f"Unexpected target values found: {unexpected_target_values}"
    )

print("Dataset validation passed.")

Dataset validation passed.


## 4. Separate Features and Target

In [5]:
X = df.drop(columns=[TARGET_COLUMN]).copy()
y = df[TARGET_COLUMN].copy()

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (1470, 30)
Target shape: (1470,)


In [6]:
target_distribution = pd.DataFrame({
    "Count": y.value_counts().sort_index(),
    "Percentage": (
        y.value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(2)
    ),
})

target_distribution.index = [
    "Stay (0)",
    "Leave (1)",
]

target_distribution

,Count,Percentage
Stay (0),1233,83.88
Leave (1),237,16.12


## 5. Identify Numerical and Categorical Features

In [7]:
numerical_features = X.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "string", "category", "bool"]
).columns.tolist()

unassigned_features = [
    column
    for column in X.columns
    if column not in numerical_features
    and column not in categorical_features
]

print(f"Numerical features: {len(numerical_features)}")
print(numerical_features)

print(f"\nCategorical features: {len(categorical_features)}")
print(categorical_features)

print(f"\nUnassigned features: {unassigned_features}")

Numerical features: 23
['Age', 'DailyRate', 'DistanceFromHome', 'Education', 'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']

Categorical features: 7
['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime']

Unassigned features: []


In [8]:
feature_type_report = pd.DataFrame({
    "Feature": X.columns,
    "Data Type": X.dtypes.astype(str).values,
    "Feature Group": [
        "Numerical"
        if column in numerical_features
        else "Categorical"
        if column in categorical_features
        else "Unassigned"
        for column in X.columns
    ],
    "Unique Values": [
        X[column].nunique(dropna=False)
        for column in X.columns
    ],
})

feature_type_report

,Feature,Data Type,Feature Group,Unique Values
0,Age,int64,Numerical,43
1,BusinessTravel,str,Categorical,3
2,DailyRate,int64,Numerical,886
3,Department,str,Categorical,3
4,DistanceFromHome,int64,Numerical,29
5,Education,int64,Numerical,5
6,EducationField,str,Categorical,6
7,EnvironmentSatisfaction,int64,Numerical,4
8,Gender,str,Categorical,2
9,HourlyRate,int64,Numerical,71


## 6. Optional Feature Engineering

The original dataset already contains many useful HR features. To keep the first production model interpretable and stable, this phase does not add speculative synthetic features.

However, we can derive a few carefully defined features that may capture employee career patterns:

- `IncomePerYearAtCompany`
- `RoleTenureRatio`
- `ManagerTenureRatio`
- `PromotionWaitRatio`

Division by zero is handled safely.

In [9]:
def add_engineered_features(data: pd.DataFrame) -> pd.DataFrame:
    engineered = data.copy()

    if {
        "MonthlyIncome",
        "YearsAtCompany",
    }.issubset(engineered.columns):
        engineered["IncomePerYearAtCompany"] = (
            engineered["MonthlyIncome"]
            / (engineered["YearsAtCompany"] + 1)
        )

    if {
        "YearsInCurrentRole",
        "YearsAtCompany",
    }.issubset(engineered.columns):
        engineered["RoleTenureRatio"] = (
            engineered["YearsInCurrentRole"]
            / (engineered["YearsAtCompany"] + 1)
        )

    if {
        "YearsWithCurrManager",
        "YearsAtCompany",
    }.issubset(engineered.columns):
        engineered["ManagerTenureRatio"] = (
            engineered["YearsWithCurrManager"]
            / (engineered["YearsAtCompany"] + 1)
        )

    if {
        "YearsSinceLastPromotion",
        "YearsAtCompany",
    }.issubset(engineered.columns):
        engineered["PromotionWaitRatio"] = (
            engineered["YearsSinceLastPromotion"]
            / (engineered["YearsAtCompany"] + 1)
        )

    return engineered

In [10]:
X_engineered = add_engineered_features(X)

new_features = [
    column
    for column in X_engineered.columns
    if column not in X.columns
]

print("Engineered features:", new_features)
print("New feature shape:", X_engineered.shape)

X_engineered[new_features].head() if new_features else "No engineered features created."

Engineered features: ['IncomePerYearAtCompany', 'RoleTenureRatio', 'ManagerTenureRatio', 'PromotionWaitRatio']
New feature shape: (1470, 34)


,IncomePerYearAtCompany,RoleTenureRatio,ManagerTenureRatio,PromotionWaitRatio
0,856.142857,0.571429,0.714286,0.000000
1,466.363636,0.636364,0.636364,0.090909
2,2090.000000,0.000000,0.000000,0.000000
3,323.222222,0.777778,0.000000,0.333333
4,1156.000000,0.666667,0.666667,0.666667


For the remainder of the notebook, the engineered feature set is used.

In [11]:
X = X_engineered

numerical_features = X.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "string", "category", "bool"]
).columns.tolist()

print("Final numerical feature count:", len(numerical_features))
print("Final categorical feature count:", len(categorical_features))

Final numerical feature count: 27
Final categorical feature count: 7


## 7. Create Train and Test Sets

`stratify=y` preserves the attrition class ratio in both training and test sets.

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (1176, 34)
X_test shape: (294, 34)
y_train shape: (1176,)
y_test shape: (294,)


In [13]:
split_distribution = pd.DataFrame({
    "Full Dataset (%)": (
        y.value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(2)
    ),
    "Training Set (%)": (
        y_train.value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(2)
    ),
    "Test Set (%)": (
        y_test.value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(2)
    ),
})

split_distribution.index = [
    "Stay (0)",
    "Leave (1)",
]

split_distribution

,Full Dataset (%),Training Set (%),Test Set (%)
Stay (0),83.88,83.84,84.01
Leave (1),16.12,16.16,15.99


## 8. Build the Numerical Preprocessing Pipeline

Numerical features use:

1. Median imputation
2. Standard scaling

Median imputation is robust to skewed distributions and outliers.

In [14]:
numerical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
    ]
)

numerical_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account fo

## 9. Build the Categorical Preprocessing Pipeline

Categorical features use:

1. Most-frequent imputation
2. One-hot encoding

`handle_unknown="ignore"` allows the API to process a valid new category structure without crashing when an unseen category appears.

In [15]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent"),
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ]
)

categorical_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('onehot', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'most_frequent'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto acc

## 10. Combine Pipelines with ColumnTransformer

In [16]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline,
            numerical_features,
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features,
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``

## 11. Fit Only on the Training Data

In [20]:
preprocessor.fit(X_train)

print("Preprocessor fitted successfully on training data only.")

Preprocessor fitted successfully on training data only.


## 12. Transform Training and Test Data

In [21]:
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)
print("Training array type:", type(X_train_processed))

Processed training shape: (1176, 55)
Processed test shape: (294, 55)
Training array type: <class 'numpy.ndarray'>


## 13. Validate Processed Data

In [22]:
if np.isnan(X_train_processed).any():
    raise ValueError(
        "Processed training data contains NaN values."
    )

if np.isnan(X_test_processed).any():
    raise ValueError(
        "Processed test data contains NaN values."
    )

if np.isinf(X_train_processed).any():
    raise ValueError(
        "Processed training data contains infinite values."
    )

if np.isinf(X_test_processed).any():
    raise ValueError(
        "Processed test data contains infinite values."
    )

print("Processed data validation passed.")

Processed data validation passed.


## 14. Inspect Generated Feature Names

In [23]:
processed_feature_names = (
    preprocessor.get_feature_names_out()
)

print(
    "Number of processed features:",
    len(processed_feature_names),
)

processed_feature_names[:30]

Number of processed features: 55


array(['numerical__Age', 'numerical__DailyRate',
       'numerical__DistanceFromHome', 'numerical__Education',
       'numerical__EnvironmentSatisfaction', 'numerical__HourlyRate',
       'numerical__JobInvolvement', 'numerical__JobLevel',
       'numerical__JobSatisfaction', 'numerical__MonthlyIncome',
       'numerical__MonthlyRate', 'numerical__NumCompaniesWorked',
       'numerical__PercentSalaryHike', 'numerical__PerformanceRating',
       'numerical__RelationshipSatisfaction',
       'numerical__StockOptionLevel', 'numerical__TotalWorkingYears',
       'numerical__TrainingTimesLastYear', 'numerical__WorkLifeBalance',
       'numerical__YearsAtCompany', 'numerical__YearsInCurrentRole',
       'numerical__YearsSinceLastPromotion',
       'numerical__YearsWithCurrManager',
       'numerical__IncomePerYearAtCompany', 'numerical__RoleTenureRatio',
       'numerical__ManagerTenureRatio', 'numerical__PromotionWaitRatio',
       'categorical__BusinessTravel_Non-Travel',
       'categoric

In [24]:
processed_feature_report = pd.DataFrame({
    "Feature Index": range(
        len(processed_feature_names)
    ),
    "Processed Feature": processed_feature_names,
})

processed_feature_report.head(30)

,Feature Index,Processed Feature
0,0,numerical__Age
1,1,numerical__DailyRate
2,2,numerical__DistanceFromHome
3,3,numerical__Education
4,4,numerical__EnvironmentSatisfaction
5,5,numerical__HourlyRate
6,6,numerical__JobInvolvement
7,7,numerical__JobLevel
8,8,numerical__JobSatisfaction
9,9,numerical__MonthlyIncome


## 15. Preview Processed Data

In [25]:
processed_train_df = pd.DataFrame(
    X_train_processed,
    columns=processed_feature_names,
    index=X_train.index,
)

processed_train_df.head()

,numerical__Age,numerical__DailyRate,numerical__DistanceFromHome,numerical__Education,numerical__EnvironmentSatisfaction,numerical__HourlyRate,numerical__JobInvolvement,numerical__JobLevel,numerical__JobSatisfaction,numerical__MonthlyIncome,numerical__MonthlyRate,numerical__NumCompaniesWorked,numerical__PercentSalaryHike,numerical__PerformanceRating,numerical__RelationshipSatisfaction,numerical__StockOptionLevel,numerical__TotalWorkingYears,numerical__TrainingTimesLastYear,numerical__WorkLifeBalance,numerical__YearsAtCompany,numerical__YearsInCurrentRole,numerical__YearsSinceLastPromotion,numerical__YearsWithCurrManager,numerical__IncomePerYearAtCompany,numerical__RoleTenureRatio,numerical__ManagerTenureRatio,numerical__PromotionWaitRatio,categorical__BusinessTravel_Non-Travel,categorical__BusinessTravel_Travel_Frequently,categorical__BusinessTravel_Travel_Rarely,categorical__Department_Human Resources,categorical__Department_Research & Development,categorical__Department_Sales,categorical__EducationField_Human Resources,categorical__EducationField_Life Sciences,categorical__EducationField_Marketing,categorical__EducationField_Medical,categorical__EducationField_Other,categorical__EducationField_Technical Degree,categorical__Gender_Female,categorical__Gender_Male,categorical__JobRole_Healthcare Representative,categorical__JobRole_Human Resources,categorical__JobRole_Laboratory Technician,categorical__JobRole_Manager,categorical__JobRole_Manufacturing Director,categorical__JobRole_Research Director,categorical__JobRole_Research Scientist,categorical__JobRole_Sales Executive,categorical__JobRole_Sales Representative,categorical__MaritalStatus_Divorced,categorical__MaritalStatus_Married,categorical__MaritalStatus_Single,categorical__OverTime_No,categorical__OverTime_Yes
1194,1.090194,1.049455,-0.899915,1.064209,-0.658710,-0.908436,1.795282,1.762189,-0.647997,2.026752,0.931289,1.330763,-0.337129,-0.432065,0.240218,2.613100,2.261482,-0.605389,0.337621,-0.665706,-0.625365,-0.368024,-0.616406,2.002002,0.063739,0.097863,0.053482,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
128,-1.634828,-0.523449,-0.899915,-1.855332,0.260202,1.694111,0.373564,-0.986265,1.153526,-0.864408,0.682742,-1.083704,-0.337129,-0.432065,0.240218,0.247430,-1.072675,-0.605389,0.337621,-0.830071,-0.905635,-0.056884,-0.897047,-0.239948,-0.546063,-0.507169,1.597700,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
810,0.981193,-0.992080,-0.777610,-1.855332,-1.577622,-0.662913,0.373564,1.762189,0.252765,2.347706,0.167705,0.123529,-0.880974,-0.432065,1.160403,0.247430,1.492061,0.190962,0.337621,0.813578,1.336527,0.565398,1.348076,0.117442,0.767357,0.795976,0.267297,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
478,-1.307825,-0.453653,0.445433,-1.855332,-0.658710,-1.252169,0.373564,-0.986265,0.252765,-0.956202,1.667056,-0.681293,-1.152896,-0.432065,0.240218,-0.935405,-0.559727,-1.401740,0.337621,-0.008246,-0.064824,-0.679165,0.506155,-0.651778,0.063739,1.005410,-0.873049,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
491,0.654191,0.491086,-0.043784,2.037390,1.179114,0.319180,0.373564,-0.070114,0.252765,-0.185956,0.728362,0.123529,-0.609051,-0.432065,-0.679966,0.247430,-0.175017,0.190962,0.337621,0.156119,0.775986,0.565398,0.786795,-0.389317,1.080077,1.106249,0.774117,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


In [ ]:
processed_train_df.describe().T.head(25)

## 16. Check One-Hot Encoded Categories

In [26]:
categorical_transformer = (
    preprocessor
    .named_transformers_["categorical"]
)

onehot_encoder = (
    categorical_transformer
    .named_steps["onehot"]
)

category_summary = {
    feature: categories.tolist()
    for feature, categories
    in zip(
        categorical_features,
        onehot_encoder.categories_,
    )
}

for feature, categories in category_summary.items():
    print("=" * 70)
    print(feature)
    print(categories)

BusinessTravel
['Non-Travel', 'Travel_Frequently', 'Travel_Rarely']
Department
['Human Resources', 'Research & Development', 'Sales']
EducationField
['Human Resources', 'Life Sciences', 'Marketing', 'Medical', 'Other', 'Technical Degree']
Gender
['Female', 'Male']
JobRole
['Healthcare Representative', 'Human Resources', 'Laboratory Technician', 'Manager', 'Manufacturing Director', 'Research Director', 'Research Scientist', 'Sales Executive', 'Sales Representative']
MaritalStatus
['Divorced', 'Married', 'Single']
OverTime
['No', 'Yes']


## 17. Save the Fitted Preprocessor

The saved object includes:

- Missing-value handling
- Numerical scaling
- One-hot encoding
- Original feature assignments

It can later be placed inside each model pipeline or used directly by the API.

In [27]:
joblib.dump(
    preprocessor,
    PREPROCESSOR_PATH,
)

print(
    "Preprocessor saved successfully to:",
    PREPROCESSOR_PATH.resolve(),
)

Preprocessor saved successfully to: C:\Users\Nasteho Abdi\employee-attrition-prediction\models\preprocessor.joblib


## 18. Save Processed Feature Names

In [28]:
processed_feature_report.to_csv(
    FEATURE_NAMES_PATH,
    index=False,
)

print(
    "Processed feature names saved to:",
    FEATURE_NAMES_PATH.resolve(),
)

Processed feature names saved to: C:\Users\Nasteho Abdi\employee-attrition-prediction\reports\processed_feature_names.csv


## 19. Save Preprocessing Metadata

In [29]:
preprocessing_metadata = {
    "source_file": DATA_PATH.name,
    "target_column": TARGET_COLUMN,
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "stratified_split": True,
    "original_rows": int(df.shape[0]),
    "original_feature_count": int(
        df.shape[1] - 1
    ),
    "engineered_features": new_features,
    "final_raw_feature_count": int(
        X.shape[1]
    ),
    "numerical_features": numerical_features,
    "categorical_features": categorical_features,
    "processed_feature_count": int(
        len(processed_feature_names)
    ),
    "train_rows": int(X_train.shape[0]),
    "test_rows": int(X_test.shape[0]),
    "train_target_counts": {
        str(key): int(value)
        for key, value
        in y_train.value_counts()
        .sort_index()
        .items()
    },
    "test_target_counts": {
        str(key): int(value)
        for key, value
        in y_test.value_counts()
        .sort_index()
        .items()
    },
    "numerical_pipeline": [
        "SimpleImputer(strategy='median')",
        "StandardScaler()",
    ],
    "categorical_pipeline": [
        "SimpleImputer(strategy='most_frequent')",
        "OneHotEncoder(handle_unknown='ignore')",
    ],
    "preprocessor_file": PREPROCESSOR_PATH.name,
}

with SPLIT_METADATA_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        preprocessing_metadata,
        file,
        indent=4,
    )

print(
    "Preprocessing metadata saved to:",
    SPLIT_METADATA_PATH.resolve(),
)

Preprocessing metadata saved to: C:\Users\Nasteho Abdi\employee-attrition-prediction\reports\preprocessing_metadata.json


## 20. Reload and Verify the Saved Preprocessor

In [30]:
loaded_preprocessor = joblib.load(
    PREPROCESSOR_PATH
)

sample_rows = X_test.head(5)

sample_transformed = (
    loaded_preprocessor.transform(sample_rows)
)

assert sample_transformed.shape[0] == 5
assert sample_transformed.shape[1] == len(
    processed_feature_names
)

print("Saved preprocessor verification passed.")
print("Sample transformed shape:", sample_transformed.shape)

Saved preprocessor verification passed.
Sample transformed shape: (5, 55)


# Phase 4 Summary

Feature engineering and preprocessing were completed successfully.

## Completed Steps

- Loaded and validated the cleaned dataset
- Separated the predictors and binary target
- Added carefully defined career-ratio features
- Identified numerical and categorical columns
- Created an 80/20 stratified train/test split
- Built a numerical pipeline with median imputation and standard scaling
- Built a categorical pipeline with most-frequent imputation and one-hot encoding
- Combined both pipelines using `ColumnTransformer`
- Fitted preprocessing only on training data
- Transformed and validated both training and test sets
- Saved the fitted preprocessor
- Saved processed feature names and metadata

## Generated Files

```text
models/
└── preprocessor.joblib

reports/
├── preprocessing_metadata.json
└── processed_feature_names.csv
```

